In [1]:
import csv
import logging
import time
from pathlib import Path
from urllib.parse import urlparse

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/thaihometown')
START_URL = "https://www.thaihometown.com/"
OUTPUT_CSV_FILE = BASE_DIR / 'thaihometown_listing_urls.csv'
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_optimized_driver():
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    return uc.Chrome(options=options, version_main=145)


def extract_links(driver, base_url):
    hrefs = driver.execute_script(
        "return [...new Set(Array.from(document.querySelectorAll('a[href]')).map(a=>a.getAttribute('href')).filter(h=>h&&h.includes('/')))]"
    )
    return [f"{base_url}{h}" if h.startswith("/") else h for h in hrefs]


def scroll_to_bottom(driver):
    last_h = 0
    for _ in range(5):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.3)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    parsed = urlparse(START_URL)
    base_url = f"{parsed.scheme}://{parsed.netloc}"

    try:
        logging.info(f"Starting scrape at: {START_URL}")
        driver.get(START_URL)

        try:
            wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
        except TimeoutException:
            return

        scroll_to_bottom(driver)

        urls = extract_links(driver, base_url)

        if not urls:
            logging.info("No links found.")
            return

        logging.info(f"Found {len(urls)} URLs")

        with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['ListingURL'])
            writer.writerows((u,) for u in sorted(urls))

        logging.info(f"Saved to {OUTPUT_CSV_FILE}")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

2026-03-29 16:14:01,835 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:14:02,547 - INFO - Starting scrape at: https://www.thaihometown.com/
2026-03-29 16:14:14,610 - INFO - Found 4084 URLs
2026-03-29 16:14:14,644 - INFO - Saved to /home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/thaihometown/thaihometown_listing_urls.csv


In [2]:
import csv
import logging
from pathlib import Path

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/thaihometown')
INPUT_CSV = BASE_DIR / 'thaihometown_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'thaihometown_full_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-extensions")
    
    return uc.Chrome(options=options, version_main=145)


def extract_content(driver: uc.Chrome) -> str:
    data = []
    
    try:
        h1 = driver.find_element(By.TAG_NAME, 'h1')
        data.extend(["--- Listing Title ---", h1.text.strip()])
    except NoSuchElementException:
        pass

    try:
        date_el = driver.find_element(By.CLASS_NAME, 'ismDate')
        data.append(date_el.text.replace('┃', '').strip())
    except NoSuchElementException:
        pass

    try:
        table = driver.find_element(By.ID, 'isTableRight')
        rows = table.find_elements(By.TAG_NAME, 'tr')
        if rows:
            data.append("\n--- Details ---")
            for row in rows:
                tds = row.find_elements(By.TAG_NAME, 'td')
                if len(tds) >= 2:
                    label = tds[0].text.strip()
                    val = tds[1].text.replace('\n', ' ').strip()
                    if label and val:
                        data.append(f"{label} {val}")
    except NoSuchElementException:
        pass

    try:
        contact_name = driver.find_element(By.ID, 'rName')
        data.extend(["\n--- Contact ---", f"Name: {contact_name.text.strip()}"])
    except NoSuchElementException:
        pass

    try:
        desc = driver.find_element(By.TAG_NAME, 'bdo')
        if desc.text.strip():
            data.extend(["\n--- Description ---", desc.text.strip()])
    except NoSuchElementException:
        pass

    return "\n".join(data)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_driver()
    wait = WebDriverWait(driver, 8)

    try:
        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    if driver.current_url != url:
                        driver.get(url)
                    
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

2026-03-29 16:17:07,046 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:17:30,120 - ERROR - [3/4084] Timeout loading: https://truehits.net/stat.php?login=thaihometown
2026-03-29 16:17:46,483 - INFO - Progress: [10/4084] scraped.
2026-03-29 16:18:13,485 - INFO - Progress: [20/4084] scraped.
2026-03-29 16:18:37,949 - ERROR - [23/4084] Timeout loading: https://www.thaihometown.com/Premium
2026-03-29 16:18:47,857 - ERROR - [24/4084] Timeout loading: https://www.thaihometown.com/Recommended
2026-03-29 16:18:57,575 - ERROR - [25/4084] Timeout loading: https://www.thaihometown.com/SpecialExclusive
2026-03-29 16:19:09,616 - INFO - Progress: [30/4084] scraped.
2026-03-29 16:19:23,644 - INFO - Progress: [40/4084] scraped.
2026-03-29 16:19:42,831 - INFO - Progress: [50/4084] scraped.
2026-03-29 16:20:19,912 - INFO - Progress: [60/4084] scraped.
2026-03-29 16:20:30,438 - ERROR - [62/4084] Timeout loading: https://www.thaih

KeyboardInterrupt: 